# Validación cruzada anidada — GDBFNet (PC-GITA y NeuroVoz)

Notebook independiente para **probar** la validación anidada y compararla con el
protocolo plano actual. No modifica nada de tus notebooks; reutiliza tu código de
`src/` y las mismas funciones de entrenamiento de cada corpus.

**Qué compara.** El protocolo *plano* selecciona hiperparámetros, aplica *early
stopping* y fija el umbral de Youden sobre las mismas particiones en las que luego
mide → introduce optimismo. El protocolo *anidado* añade un bucle externo que solo
se usa para medir: los hiperparámetros, el *early stopping* y el umbral se deciden
únicamente en particiones internas (dentro del *train* externo), y el fold externo
permanece intacto hasta la evaluación.

**Coste.** Con codificadores congelados y embeddings precomputados, cada ajuste
cuesta ~1 s. Una pasada anidada 5×5 ronda los ~25 min (PC-GITA) y ~15 min
(NeuroVoz). Pon `QUICK = True` para una prueba rápida.

In [1]:
# =========================== CONFIGURACIÓN ===========================
import os, sys, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = "/Users/napster/Documents/upm"   # raíz de tu proyecto
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)                            # para que "data/..." resuelva

import numpy as np
import torch

CORPUS       = "neurovoz"        # "pcgita"  o  "neurovoz"
USE_ICMM     = True             # aplicar regularización intra-clase (ICMM)
TARGET_SR    = 16_000
RANDOM_STATE = 42

N_OUTER  = 5     # folds externos (evaluación insesgada)
N_INNER  = 5     # folds internos (selección de HP, umbral y ensemble)
N_TRIALS = 75 if CORPUS == "pcgita" else 40   # trials de Optuna por fold externo

QUICK = False    # True -> prueba rápida (menos trials y epochs)
if QUICK:
    N_TRIALS = 12
EPOCHS   = 40 if QUICK else 120
PATIENCE = 10 if QUICK else 20

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")

np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)
CACHE = os.path.join(PROJECT_ROOT, "nested_cache")
os.makedirs(CACHE, exist_ok=True)
print(f"Corpus={CORPUS} | ICMM={USE_ICMM} | device={DEVICE} | "
      f"{N_OUTER}x{N_INNER} nested | trials={N_TRIALS}")

Corpus=neurovoz | ICMM=True | device=mps | 5x5 nested | trials=40


## 1 — Embeddings

Se extraen una sola vez y se cachean en `nested_cache/{corpus}.npz`. Si el fichero
ya existe, se cargan al instante; si no, se reproduce el mismo pipeline de tu
notebook del corpus correspondiente.

In [2]:
# ========================= PREPARACIÓN DE EMBEDDINGS =========================
# Produce tres arrays a nivel de sujeto: emb_t_all (temporal), emb_s_all (espectral)
# y labels_all. Idéntico al de tus notebooks; solo se ha unificado y cacheado.
from sklearn.preprocessing import StandardScaler

cache_file = os.path.join(CACHE, f"{CORPUS}.npz")

def _standardize(emb_t_raw, emb_s_raw, labels):
    # Nota: se estandariza sobre todo el conjunto igual que en tus notebooks intra-corpus.
    emb_t = StandardScaler().fit_transform(emb_t_raw)
    emb_s = StandardScaler().fit_transform(emb_s_raw)
    return emb_t, emb_s, labels

if os.path.exists(cache_file):
    d = np.load(cache_file)
    emb_t_all, emb_s_all, labels_all = d["emb_t"], d["emb_s"], d["labels"]
    print(f"Cargado de cache: {cache_file}")

elif CORPUS == "pcgita":
    # ---- Pipeline PC-GITA (de dual_branch_fusion_network_experiments.ipynb) ----
    from src.preprocessing import load_waveforms, preprocess_waveform, load_metadata
    from src.embeddings import extract_multilayer_embeddings
    from src.spectral import extract_spectral_features

    df_metadata = load_metadata(base_path="data")
    waveforms_raw, df_metadata = load_waveforms(df_metadata, target_sr=TARGET_SR)
    waveforms_processed = [preprocess_waveform(wf) for wf in waveforms_raw]

    w2v_layers = extract_multilayer_embeddings(
        waveforms_processed, "facebook/wav2vec2-base", [0], DEVICE)
    emb_w2v_L0 = w2v_layers[0]

    MEL_CONFIG = {"sample_rate": TARGET_SR, "n_fft": 1024, "hop_length": 512, "n_mels": 128}
    spectral_features = extract_spectral_features(waveforms_processed, device=DEVICE, **MEL_CONFIG)

    labels = df_metadata["label"].to_numpy()
    emb_t_all, emb_s_all, labels_all = _standardize(emb_w2v_L0, spectral_features, labels)
    np.savez(cache_file, emb_t=emb_t_all, emb_s=emb_s_all, labels=labels_all)
    print(f"Extraído y cacheado: {cache_file}")

elif CORPUS == "neurovoz":
    # ---- Pipeline NeuroVoz (de neurovoz_dual_branch_resnet18.ipynb) ----
    import pandas as pd
    from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
    from src.spectral import extract_spectral_features
    import soundfile as sf
    from math import gcd
    from scipy.signal import resample_poly

    NEUROVOZ_ROOT = "data/neurovoz"
    AUDIO_DIR = f"{NEUROVOZ_ROOT}/audios"
    META_HC   = f"{NEUROVOZ_ROOT}/metadata/metadata_hc.csv"
    META_PD   = f"{NEUROVOZ_ROOT}/metadata/metadata_pd.csv"
    TASK      = "PATAKA"

    def load_neurovoz_metadata(meta_hc_path, meta_pd_path, task=TASK):
        hc = pd.read_csv(meta_hc_path); pd_ = pd.read_csv(meta_pd_path)
        df = pd.concat([hc, pd_], ignore_index=True)
        df["filename"] = df["Audio"].str.split("/").str[-1]
        df["task"] = df["filename"].str.extract(r"^[A-Z]+_(.+)_\d+\.wav")
        df = df[df["task"] == task].copy()
        df["label"] = (df["Group"] == "PD").astype(int)
        df["audio_path"] = df["filename"].apply(lambda f: os.path.join(AUDIO_DIR, f))
        return df[["label", "audio_path"]].reset_index(drop=True)

    def load_and_resample(path, target_sr=TARGET_SR):
        wav, sr = sf.read(path, dtype="float32", always_2d=False)
        if wav.ndim > 1: wav = wav.mean(axis=1)
        if sr != target_sr:
            g = gcd(sr, target_sr); wav = resample_poly(wav, target_sr // g, sr // g)
        return wav.astype(np.float32)

    def preprocess_waveform(x, pre_emphasis=0.97):
        x = x - np.mean(x)
        x = np.append(x[0], x[1:] - pre_emphasis * x[:-1])
        peak = np.max(np.abs(x))
        return x / peak if peak > 0 else x

    def extract_wav2vec2_embeddings(waveforms, model_id="facebook/wav2vec2-base", layer=0, device=DEVICE):
        extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_id)
        model = Wav2Vec2Model.from_pretrained(model_id, output_hidden_states=True).to(device).eval()
        embs = []
        with torch.no_grad():
            for wav in waveforms:
                iv = extractor(wav, sampling_rate=TARGET_SR, return_tensors="pt").input_values.to(device)
                h = model(iv).hidden_states[layer].squeeze(0)
                embs.append(h.mean(dim=0).cpu().numpy())
        return np.stack(embs)

    df_meta = load_neurovoz_metadata(META_HC, META_PD, TASK)
    waveforms = [preprocess_waveform(load_and_resample(p)) for p in df_meta["audio_path"]]
    emb_w2v = extract_wav2vec2_embeddings(waveforms, layer=0, device=DEVICE)
    spectral_features = extract_spectral_features(waveforms, device=DEVICE)

    labels = df_meta["label"].to_numpy()
    emb_t_all, emb_s_all, labels_all = _standardize(emb_w2v, spectral_features, labels)
    np.savez(cache_file, emb_t=emb_t_all, emb_s=emb_s_all, labels=labels_all)
    print(f"Extraído y cacheado: {cache_file}")

else:
    raise ValueError("CORPUS debe ser 'pcgita' o 'neurovoz'")

labels_all = labels_all.astype(np.float32)
print(f"Total: {len(labels_all)} sujetos | PD={int(labels_all.sum())} "
      f"HC={int((labels_all==0).sum())} | dim_t={emb_t_all.shape[1]} dim_s={emb_s_all.shape[1]}")

2026-07-08 22:31:42,138 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-08 22:31:42,141 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-08 22:31:42,330 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-08 22:31:42,398 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/wav2vec2-base/0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8/preprocessor_config.json "HTTP/1.1 200 OK"
2026-07-08 22:31:42,497 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-08 22:31:42,564 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/res

Extraído y cacheado: /Users/napster/Documents/upm/nested_cache/neurovoz.npz
Total: 99 sujetos | PD=49 HC=50 | dim_t=768 dim_s=512


## 2 — Bloques de entrenamiento

Réplica fiel de tus funciones. El modelo depende del corpus: **PC-GITA** usa
`DualBranchFusionNet` (con compuerta, de `src.models`) y **NeuroVoz** usa
`DualBranchClassifier` (concatenación). El bucle de entrenamiento (AdamW +
`ReduceLROnPlateau` + *label smoothing* + *early stopping*) se ha unificado, pero
se comporta igual que en tus notebooks. La diferencia frente a ellos es que aquí el
modelo entrenado se devuelve para poder predecir un fold externo distinto del de
*early stopping* (evitando fuga).

In [3]:
# ============================ MODELO Y ENTRENAMIENTO ============================
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, roc_curve

from src.models import DualBranchFusionNet   # modelo con compuerta (PC-GITA)

class DualBranchClassifier(nn.Module):
    """Concatenación (NeuroVoz). Idéntico al definido en su notebook."""
    def __init__(self, dim_t, dim_s, dim_proj=128, dim_hidden=64, dropout=0.3):
        super().__init__()
        self.proj_t = nn.Sequential(nn.Linear(dim_t, dim_proj), nn.ReLU(), nn.Dropout(dropout))
        self.proj_s = nn.Sequential(nn.Linear(dim_s, dim_proj), nn.ReLU(), nn.Dropout(dropout))
        self.head = nn.Sequential(
            nn.Linear(2 * dim_proj, dim_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dim_hidden, 1))
    def forward(self, x_t, x_s):
        z = torch.cat([self.proj_t(x_t), self.proj_s(x_s)], dim=1)
        return self.head(z).squeeze(-1)

class DualBranchDataset(Dataset):
    def __init__(self, emb_t, emb_s, labels, idx):
        self.emb_t = torch.tensor(emb_t[idx], dtype=torch.float32)
        self.emb_s = torch.tensor(emb_s[idx], dtype=torch.float32)
        self.labels = torch.tensor(labels[idx], dtype=torch.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.emb_t[i], self.emb_s[i], self.labels[i]

def make_model(hp, dim_t, dim_s):
    if CORPUS == "pcgita":
        return DualBranchFusionNet(dim_proj=hp["dim_proj"], dim_hidden=hp["dim_hidden"],
                                   dropout=hp["dropout"])
    return DualBranchClassifier(dim_t, dim_s, hp["dim_proj"], hp["dim_hidden"], hp["dropout"])

def _logits(out):
    # el modelo con compuerta devuelve (logits, alpha); el de concatenación, logits
    return out[0] if isinstance(out, tuple) else out

def fit_model(model, train_ds, val_ds, hp):
    """Entrena con early stopping sobre val_ds y devuelve el modelo en su mejor estado."""
    model = model.to(DEVICE)
    tr = DataLoader(train_ds, batch_size=hp["batch_size"], shuffle=True)
    va = DataLoader(val_ds, batch_size=len(val_ds))
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=7)
    crit = nn.BCEWithLogitsLoss()
    ls = hp.get("label_smoothing", 0.0)

    best_loss, best_state, bad = float("inf"), None, 0
    for _ in range(EPOCHS):
        model.train()
        for z_t, z_s, y in tr:
            z_t, z_s, y = z_t.to(DEVICE), z_s.to(DEVICE), y.to(DEVICE)
            if ls > 0: y = y * (1 - ls) + 0.5 * ls
            loss = crit(_logits(model(z_t, z_s)), y)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            z_t, z_s, y = next(iter(va))
            vloss = crit(_logits(model(z_t.to(DEVICE), z_s.to(DEVICE))), y.to(DEVICE)).item()
        sched.step(vloss)
        if vloss < best_loss:
            best_loss, best_state, bad = vloss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    return model.eval()

@torch.no_grad()
def predict(model, emb_t, emb_s, idx):
    z_t = torch.tensor(emb_t[idx], dtype=torch.float32, device=DEVICE)
    z_s = torch.tensor(emb_s[idx], dtype=torch.float32, device=DEVICE)
    return torch.sigmoid(_logits(model(z_t, z_s))).cpu().numpy()

def manifold_mixup_intraclass(emb_t, emb_s, labels, alpha, rng):
    """Un ejemplo interpolado intra-clase por muestra (ICMM). Idéntico al de NeuroVoz."""
    aug_t, aug_s, aug_l = [], [], []
    for cls in [0, 1]:
        cls_idx = np.where(labels == cls)[0]
        if len(cls_idx) < 2: continue
        for i in cls_idx:
            j = rng.choice(cls_idx[cls_idx != i])
            lam = rng.beta(alpha, alpha)
            aug_t.append(lam * emb_t[i] + (1 - lam) * emb_t[j])
            aug_s.append(lam * emb_s[i] + (1 - lam) * emb_s[j])
            aug_l.append(cls)
    return np.array(aug_t), np.array(aug_s), np.array(aug_l)

def youden_threshold(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return thr[np.argmax(tpr - fpr)]

def metrics_at(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(int)
    return dict(auc=roc_auc_score(y_true, y_prob),
                acc=accuracy_score(y_true, y_pred),
                sens=recall_score(y_true, y_pred, pos_label=1, zero_division=0),
                spec=recall_score(y_true, y_pred, pos_label=0, zero_division=0))

def bootstrap_ci(vals, n_boot=1000, ci=95, seed=42):
    rng = np.random.default_rng(seed); vals = np.asarray(vals)
    boots = [rng.choice(vals, len(vals), replace=True).mean() for _ in range(n_boot)]
    return vals.mean(), np.percentile(boots, (100-ci)/2), np.percentile(boots, 100-(100-ci)/2)

## 3 — Espacio de búsqueda y ajuste de un fold

`train_eval` entrena en un conjunto de índices (aplicando ICMM si procede,
solo sobre el train) con *early stopping* en `val_idx`, y predice `eval_idx`.

In [4]:
# ============================ BÚSQUEDA E INFERENCIA ============================
import optuna
from sklearn.model_selection import StratifiedKFold
optuna.logging.set_verbosity(optuna.logging.WARNING)

DIM_T, DIM_S = emb_t_all.shape[1], emb_s_all.shape[1]

def search_space(trial):
    hp = dict(
        lr=trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        weight_decay=trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
        dropout=trial.suggest_float("dropout", 0.1, 0.5, step=0.05),
        label_smoothing=trial.suggest_float("label_smoothing", 0.0, 0.15, step=0.025),
        batch_size=trial.suggest_categorical("batch_size", [8, 16, 32]),
        dim_proj=trial.suggest_categorical("dim_proj", [64, 128, 256]),
        dim_hidden=trial.suggest_categorical("dim_hidden", [32, 64, 128]),
    )
    if USE_ICMM:
        hp["mixup_alpha"] = trial.suggest_float("mixup_alpha", 0.1, 1.0, step=0.1)
    return hp

def train_eval(train_idx, val_idx, eval_idx, hp):
    """Entrena en train_idx (con ICMM opcional), early stopping en val_idx, predice eval_idx."""
    if USE_ICMM and hp.get("mixup_alpha", 0) > 0:
        rng = np.random.default_rng(RANDOM_STATE)
        at, as_, al = manifold_mixup_intraclass(
            emb_t_all[train_idx], emb_s_all[train_idx], labels_all[train_idx], hp["mixup_alpha"], rng)
        full_t = np.concatenate([emb_t_all[train_idx], at]) if len(at) else emb_t_all[train_idx]
        full_s = np.concatenate([emb_s_all[train_idx], as_]) if len(at) else emb_s_all[train_idx]
        full_l = np.concatenate([labels_all[train_idx], al]) if len(at) else labels_all[train_idx]
        train_ds = DualBranchDataset(full_t, full_s, full_l, np.arange(len(full_l)))
    else:
        train_ds = DualBranchDataset(emb_t_all, emb_s_all, labels_all, train_idx)
    val_ds = DualBranchDataset(emb_t_all, emb_s_all, labels_all, val_idx)
    model = fit_model(make_model(hp, DIM_T, DIM_S), train_ds, val_ds, hp)
    return predict(model, emb_t_all, emb_s_all, eval_idx)

def optuna_search(idx, n_trials):
    """Selecciona HP por CV interna sobre el subconjunto idx (nunca ve el fold externo)."""
    y = labels_all[idx]
    def objective(trial):
        hp = search_space(trial)
        inner = StratifiedKFold(N_INNER, shuffle=True, random_state=RANDOM_STATE)
        aucs = []
        for itr, iva in inner.split(idx, y):
            yp = train_eval(idx[itr], idx[iva], idx[iva], hp)
            aucs.append(roc_auc_score(labels_all[idx[iva]], yp))
        return np.mean(aucs) - 0.5 * np.std(aucs)
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study.best_params

## 4 — Validación cruzada anidada

Por cada fold externo: (1) Optuna sobre el *train* externo, (2) umbral de Youden a
partir de las predicciones OOF internas, (3) predicción del fold externo por
*ensemble* de los modelos internos. El fold externo no interviene en (1)-(2).

In [5]:
# ================================ NESTED CV ================================
import time
t0 = time.time()

outer = StratifiedKFold(N_OUTER, shuffle=True, random_state=RANDOM_STATE)
nested_folds = []                 # métricas por fold externo
oof_true, oof_prob = [], []       # predicciones fuera de muestra agregadas

for k, (otr, ote) in enumerate(outer.split(emb_t_all, labels_all), 1):
    bp = optuna_search(otr, N_TRIALS)                      # (1) HP honestos

    inner = StratifiedKFold(N_INNER, shuffle=True, random_state=RANDOM_STATE)
    inner_true, inner_prob, test_probs = [], [], []
    for itr, iva in inner.split(otr, labels_all[otr]):
        inner_prob.append(train_eval(otr[itr], otr[iva], otr[iva], bp))  # OOF interno (umbral)
        inner_true.append(labels_all[otr[iva]])
        test_probs.append(train_eval(otr[itr], otr[iva], ote, bp))       # predicción del fold externo
    inner_true = np.concatenate(inner_true); inner_prob = np.concatenate(inner_prob)

    thr = youden_threshold(inner_true, inner_prob)         # (2) umbral sin ver el externo
    prob_ote = np.mean(test_probs, axis=0)                 # (3) ensemble sobre el externo
    m = metrics_at(labels_all[ote], prob_ote, thr)
    nested_folds.append(m)
    oof_true.append(labels_all[ote]); oof_prob.append(prob_ote)
    print(f"  Fold externo {k}: AUC={m['auc']:.3f} Acc={m['acc']:.3f} "
          f"Sens={m['sens']:.3f} Spec={m['spec']:.3f} (thr={thr:.2f})")

oof_true = np.concatenate(oof_true); oof_prob = np.concatenate(oof_prob)
print(f"\nNested CV completado en {(time.time()-t0)/60:.1f} min")

Best trial: 37. Best value: 0.901663: 100%|██████████| 40/40 [01:37<00:00,  2.44s/it]


  Fold externo 1: AUC=0.860 Acc=0.800 Sens=0.900 Spec=0.700 (thr=0.34)


Best trial: 31. Best value: 0.956927: 100%|██████████| 40/40 [01:55<00:00,  2.89s/it]


  Fold externo 2: AUC=0.810 Acc=0.750 Sens=0.800 Spec=0.700 (thr=0.53)


Best trial: 8. Best value: 0.925963: 100%|██████████| 40/40 [01:33<00:00,  2.35s/it]


  Fold externo 3: AUC=0.980 Acc=0.900 Sens=0.900 Spec=0.900 (thr=0.66)


Best trial: 34. Best value: 0.923619: 100%|██████████| 40/40 [01:55<00:00,  2.89s/it]


  Fold externo 4: AUC=0.980 Acc=0.800 Sens=1.000 Spec=0.600 (thr=0.40)


Best trial: 39. Best value: 0.917628: 100%|██████████| 40/40 [01:34<00:00,  2.36s/it]


  Fold externo 5: AUC=0.889 Acc=0.737 Sens=0.444 Spec=1.000 (thr=0.71)

Nested CV completado en 8.9 min


## 5 — Protocolo plano (para comparar)

Reproduce tu esquema actual: una única búsqueda de HP sobre todos los datos y un
CV de 5 folds donde el umbral se fija en cada fold de validación (el mismo que se
mide). Es el que produce tus cifras publicadas.

In [6]:
# ============================ FLAT (tu protocolo actual) ============================
all_idx = np.arange(len(labels_all))
bp_flat = optuna_search(all_idx, N_TRIALS)

flat = StratifiedKFold(N_OUTER, shuffle=True, random_state=RANDOM_STATE)
flat_folds = []
for tr, va in flat.split(emb_t_all, labels_all):
    yp = train_eval(tr, va, va, bp_flat)          # umbral y medida sobre el MISMO fold
    thr = youden_threshold(labels_all[va], yp)
    flat_folds.append(metrics_at(labels_all[va], yp, thr))
print("Flat CV completado.")

Best trial: 29. Best value: 0.932599: 100%|██████████| 40/40 [03:06<00:00,  4.66s/it]


Flat CV completado.


In [7]:
# ================================ COMPARACIÓN ================================
def summ(folds, key):
    v = np.array([f[key] for f in folds]); return v.mean(), v.std()

print("=" * 66)
print(f"  {CORPUS.upper()}  |  ICMM={USE_ICMM}  |  nested {N_OUTER}x{N_INNER}, trials={N_TRIALS}")
print("=" * 66)
print(f"  {'Métrica':<14}{'Plano (media±sd)':>22}{'Anidado (media±sd)':>24}")
print("  " + "-" * 62)
for key, name in zip(["auc","acc","sens","spec"], ["AUC","Accuracy","Sensibilidad","Especificidad"]):
    fm, fs = summ(flat_folds, key); nm, ns = summ(nested_folds, key)
    print(f"  {name:<14}{fm:>10.3f} ± {fs:<7.3f}{nm:>12.3f} ± {ns:<7.3f}")

m, lo, hi = bootstrap_ci([f["auc"] for f in nested_folds])
auc_oof = roc_auc_score(oof_true, oof_prob)
print("  " + "-" * 62)
print(f"  AUC anidado (IC 95% bootstrap sobre folds): {m:.3f} [{lo:.3f}, {hi:.3f}]")
print(f"  AUC anidado agregando predicciones OOF:     {auc_oof:.3f}")
print(f"  Brecha de optimismo (AUC plano - anidado):  "
      f"{summ(flat_folds,'auc')[0] - summ(nested_folds,'auc')[0]:+.3f}")

  NEUROVOZ  |  ICMM=True  |  nested 5x5, trials=40
  Métrica             Plano (media±sd)      Anidado (media±sd)
  --------------------------------------------------------------
  AUC                0.941 ± 0.038         0.904 ± 0.067  
  Accuracy           0.909 ± 0.038         0.797 ± 0.057  
  Sensibilidad       0.900 ± 0.063         0.809 ± 0.193  
  Especificidad      0.920 ± 0.098         0.780 ± 0.147  
  --------------------------------------------------------------
  AUC anidado (IC 95% bootstrap sobre folds): 0.904 [0.846, 0.962]
  AUC anidado agregando predicciones OOF:     0.887
  Brecha de optimismo (AUC plano - anidado):  +0.038
